In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Porovnání nastavení transpileru

<Accordion>
<AccordionItem title="Package versions">

The code on this page was developed using the following requirements.
We recommend using these versions or newer.

```
qiskit[all]~=2.4.0
qiskit-ibm-runtime~=0.46.1
```
</AccordionItem>
</Accordion>
Různá nastavení Transpileru poskytují různé typy optimalizace Circuit, často na úkor delší doby klasického zpracování. Tento průvodce projde celým procesem vytváření, transpilování a odesílání Circuit a ukáže testování výkonnosti různých nastavení.

Stejné nastavení může zlepšit výsledky jednoho Circuit, zatímco jinému může uškodit. Před spuštěním na skutečném hardwaru si vždy zkontroluj výsledné transpilované Circuit.
## Nastavení a vytvoření ukázkového Circuit

In [1]:
# Create circuit to test transpiler on
from qiskit import QuantumCircuit
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit.circuit.library import grover_operator, DiagonalGate

# Use Statevector object to calculate the ideal output
from qiskit.quantum_info import Statevector
from qiskit.visualization import plot_histogram
from qiskit.transpiler import PassManager

from qiskit.circuit.library import XGate
from qiskit.quantum_info import hellinger_fidelity

Vytvoř malý Circuit, na němž se Transpiler pokusí provést optimalizaci. Tento příklad vytváří Circuit, který provádí Groverův algoritmus s orákulem, které označuje stav `111`. Poté simuluj ideální distribuci (co bys očekával/a naměřit, kdybys toto spustil/a na dokonalém kvantovém počítači nekonečněkrát) pro pozdější porovnání.

In [2]:
oracle = DiagonalGate([1] * 7 + [-1])
qc = QuantumCircuit(3)
qc.h([0, 1, 2])
qc = qc.compose(grover_operator(oracle))

qc.draw(output="mpl", style="iqp")

<Image src="../docs/images/guides/circuit-transpilation-settings/extracted-outputs/4ac958d4-b9b5-4939-a359-a9edca7ddb6a-0.svg" alt="Output of the previous code cell" />

In [3]:
ideal_distribution = Statevector.from_instruction(qc).probabilities_dict()

plot_histogram(ideal_distribution)

<Image src="../docs/images/guides/circuit-transpilation-settings/extracted-outputs/6313186e-bc40-432e-9ada-8594d6a26d55-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/circuit-transpilation-settings/extracted-outputs/6313186e-bc40-432e-9ada-8594d6a26d55-0.svg)

## Transpilace
Dále transpiluj Circuit pro QPU. Porovnáš výkonnost Transpileru s `optimization_level` nastaveným na `0` (nejnižší) oproti `3` (nejvyšší). Nejnižší úroveň optimalizace dělá naprosté minimum potřebné pro spuštění Circuit na zařízení – mapuje Qubity Circuit na Qubity zařízení a přidává swap Gate, aby bylo možné provádět všechny dvoQubitové operace. Nejvyšší úroveň optimalizace je mnohem chytřejší a využívá řadu triků ke snížení celkového počtu Gate. Protože vícQubitové Gate mají vysokou míru chyb a Qubity se dekoherují v čase, kratší Circuit by měly přinést lepší výsledky.

> **Important:** Tento příklad používá hardware IBM Quantum&reg;, ale můžeš ho vyzkoušet na libovolném QPU kompatibilním s Qiskit. Výsledky se mohou lišit.

Následující buňka transpiluje `qc` pro obě hodnoty `optimization_level`, vypíše počet dvoQubitových Gate a přidá transpilované Circuit do seznamu. Některé algoritmy Transpileru jsou náhodné, proto se nastavuje seed pro reprodukovatelnost.

In [4]:
# Use Qiskit Runtime to run jobs on hardware
from qiskit_ibm_runtime import (
    QiskitRuntimeService,
    SamplerV2 as Sampler,
)

In [5]:
# Select the backend with the fewest number of jobs in the queue
service = QiskitRuntimeService()
backend = service.least_busy(
    operational=True, simulator=False, min_num_qubits=127
)
backend.name

'ibm_marrakesh'

In [6]:
# Need to add measurements to the circuit
qc.measure_all()

# Find the correct two-qubit gate
twoQ_gates = set(["ecr", "cz", "cx"])
for gate in backend.basis_gates:
    if gate in twoQ_gates:
        twoQ_gate = gate

circuits = []
for optimization_level in [0, 3]:
    pm = generate_preset_pass_manager(
        optimization_level, backend=backend, seed_transpiler=0
    )
    t_qc = pm.run(qc)
    print(
        f"Two-qubit gates (optimization_level={optimization_level}): ",
        t_qc.count_ops()[twoQ_gate],
    )
    circuits.append(t_qc)

Two-qubit gates (optimization_level=0):  21
Two-qubit gates (optimization_level=3):  12


Since CNOTs usually have a high error rate, the circuit transpiled with `optimization_level=3` should perform much better.

Another way you can improve performance is through [dynamical decoupling](/docs/api/qiskit/qiskit.transpiler.passes.PadDynamicalDecoupling), by applying a sequence of gates to idling qubits. This cancels out some unwanted interactions with the environment. The following cell adds dynamic decoupling to the circuit transpiled with `optimization_level=3` and adds it to the list.

In [7]:
from qiskit_ibm_runtime.transpiler.passes.scheduling import (
    ASAPScheduleAnalysis,
    PadDynamicalDecoupling,
)

# Get gate durations so the transpiler knows how long each operation takes
durations = backend.target.durations()

# This is the sequence we'll apply to idling qubits
dd_sequence = [XGate(), XGate()]

# Run scheduling and dynamic decoupling passes on circuit
pm = PassManager(
    [
        ASAPScheduleAnalysis(durations),
        PadDynamicalDecoupling(durations, dd_sequence),
    ]
)
circ_dd = pm.run(circuits[1])

# Add this new circuit to our list
circuits.append(circ_dd)

In [8]:
circ_dd.draw(output="mpl", style="iqp", idle_wires=False)

<Image src="../docs/images/guides/circuit-transpilation-settings/extracted-outputs/c1c91fbd-acfe-413e-a6c9-ad97f4dd5543-0.svg" alt="Output of the previous code cell" />

Protože CNOT Gate mívají obvykle vysokou míru chyb, Circuit transpilovaný s `optimization_level=3` by měl dosahovat výrazně lepších výsledků.

Dalším způsobem, jak zlepšit výkon, je [dynamické oddělování](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.PadDynamicalDecoupling), při němž se aplikuje sekvence Gate na nečinné Qubity. Tím se potlačují některé nežádoucí interakce s prostředím. Následující buňka přidá dynamické oddělování do obvod transpilovaného s `optimization_level=3` a přidá ho do seznamu.

In [9]:
sampler = Sampler(backend)
job = sampler.run(
    [(circuit) for circuit in circuits],  # sample all three circuits
    shots=8000,
)
result = job.result()

## View results

Finally, plot the results from the device runs against the ideal distribution. You can see the results with `optimization_level=3` are closer to the ideal distribution due to the lower gate count, and `optimization_level=3 + dd` is even closer due to the dynamical decoupling.

In [10]:
binary_prob = [
    {
        k: v / res.data.meas.num_shots
        for k, v in res.data.meas.get_counts().items()
    }
    for res in result
]
plot_histogram(
    binary_prob + [ideal_distribution],
    bar_labels=False,
    legend=[
        "optimization_level=0",
        "optimization_level=3",
        "optimization_level=3 + dd",
        "ideal distribution",
    ],
)

<Image src="../docs/images/guides/circuit-transpilation-settings/extracted-outputs/9e86132d-a8b2-40db-af42-53042dfa108b-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/circuit-transpilation-settings/extracted-outputs/c1c91fbd-acfe-413e-a6c9-ad97f4dd5543-0.svg)

## Spuštění Circuit
V tomto okamžiku máš seznam Circuit transpilovaných s různými nastaveními. Dále spusť tyto Circuit pomocí primitiva Sampler a ulož výsledky do `result`.

In [11]:
for prob in binary_prob:
    print(f"{hellinger_fidelity(prob, ideal_distribution):.3f}")

0.985
0.989
0.988


## Zobrazení výsledků
Nakonec vykresli výsledky ze spuštění na zařízení oproti ideální distribuci. Výsledky s `optimization_level=3` jsou blíže ideální distribuci díky nižšímu počtu Gate, a `optimization_level=3 + dd` je ještě blíže díky dynamickému oddělování.